In [4]:
from dotenv import load_dotenv
load_dotenv("../.env")


True

In [33]:

import json
import os
from no_tool_agent_runner import getFilterDeviceRunner 
from sr_app_types.no_tool_agent_types import State

# === 設定 ===
FOV_PATH = "../Evaluation/TestData/en/long/fov.json"
RESULT_DIR = "../Evaluation/Result"
RESULT_FILE = os.path.join(RESULT_DIR, "fov_test_results.json")
MAX_TESTS = 10

# === データ読み込み ===
with open(FOV_PATH, "r", encoding="utf-8") as f:
    fov_data = json.load(f)

# === 結果保存ディレクトリ作成 ===
os.makedirs(RESULT_DIR, exist_ok=True)


In [56]:

import uuid# === UUIDでid付与 ===
# === 各エントリにUUIDでid付与 ===
for item in fov_data:
    if "id" not in item:
        item["id"] = str(uuid.uuid4())

# === JSONを上書き保存 ===
with open(FOV_PATH, "w", encoding="utf-8") as f:
    json.dump(fov_data, f, ensure_ascii=False, indent=2)

In [34]:
fov_data[0]

{'user_prompt': 'Turn on all visible ones in my field of view within 2.0 meters.',
 'output': {'filter_type': 'fov',
  'params': {'isInFov': True, 'order': 'proximity', 'range': 2.0}}}

In [52]:

# === 初期化 ===
runner = getFilterDeviceRunner()
results = []
test_count = 0
success_count = 0

# === テスト実行 ===
# === テスト実行 ===
# for i, item in enumerate(fov_data[:MAX_TESTS]):
i = 0 
item = fov_data[0]
user_prompt = item["user_prompt"]
expected_filter_type = item["output"]["filter_type"]
expected_params = item["output"]["params"]

print(f"\n===== FOVテスト {i+1}/{MAX_TESTS} =====")
print(f"🗣 user_prompt: {user_prompt}")





===== FOVテスト 1/10 =====
🗣 user_prompt: Turn on all visible ones in my field of view within 2.0 meters.


In [ ]:
# Agent 実行
state = State(user_prompt=user_prompt)
res = runner.invoke(state)

In [64]:
res['filterAgent'].selected_tool

{'filter_type': 'fov',
 'params': {'isInFov': True, 'order': 'proximity', 'range': 2.0}}

In [37]:
res["filterAgent"].metrics["cost_usd"] * 	138

1.82988

In [42]:
selected_tool = res['filterAgent'].selected_tool
selected_filter_type = selected_tool.get("filter_type", None)
selected_filter_param = selected_tool.get("params", {})

test_type = item["output"]["filter_type"]

In [47]:
def exclude_order(params):
    return {k: v for k, v in params.items() if k != "order"}


expected_clean = exclude_order(expected_params)
actual_clean = exclude_order(selected_filter_param)


params_match = (expected_clean == actual_clean)
filter_type_match = (selected_filter_type == item["output"]["filter_type"])
print(filter_type_match)

True


In [40]:
item 

{'user_prompt': 'Turn on all visible ones in my field of view within 2.0 meters.',
 'output': {'filter_type': 'fov',
  'params': {'isInFov': True, 'order': 'proximity', 'range': 2.0}}}

In [53]:
selected_tool = res['filterAgent'].selected_tool
actual_filter_type = selected_tool.get("filter_type", None)
actual_params = selected_tool.get("params", {})

# 評価
type_match = (expected_filter_type == actual_filter_type)
params_match = (expected_params == actual_params)
overall_match = type_match and params_match

test_count += 1
if overall_match:
    success_count += 1

print(f"✅ expected: {expected_filter_type}, {expected_params}")
print(f"🏁 actual:   {actual_filter_type}, {actual_params}")
print(f"🎯 filter_type一致: {type_match}")
print(f"🎯 params一致:      {params_match}")
print(f"🎯 総合評価: {'成功 ✓' if overall_match else '失敗 ✗'}")

# メトリクス取得
token_info = res["filterAgent"].metrics["tokens"]
cost_info = res["filterAgent"].metrics["cost_usd"]
time_info = res["filterAgent"].metrics["elapsed_seconds"]

# 結果保存
results.append({
    "index": i,
    "user_prompt": user_prompt,
    "expected": {
        "filter_type": expected_filter_type,
        "params": expected_params
    },
    "actual": {
        "filter_type": actual_filter_type,
        "params": actual_params
    },
    "evaluation": {
        "filter_type_match": type_match,
        "params_match": params_match,
        "overall_match": overall_match
    },
    "metrics": {
        "tokens": token_info,
        "cost_usd": cost_info,
        "elapsed_seconds": time_info
    }
})


✅ expected: fov, {'isInFov': True, 'order': 'proximity', 'range': 2.0}
🏁 actual:   fov, {'isInFov': True, 'order': 'proximity', 'range': 2.0}
🎯 filter_type一致: True
🎯 params一致:      True
🎯 総合評価: 成功 ✓


In [54]:
test_count

1